<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Building the Demo

In the previous notebooks you built the individual pieces of the lane-following pipeline, and in
[notebook 05](05_finite_state_machine.ipynb) you saw how the **Finite State Machine** decides which nodes
are active. In this final notebook we tie everything together into a runnable **demo**.

A "demo" in Duckietown is the top-level thing you launch on a robot with the `dts devel` workflow from
[notebook 00](00_DTS_devel_API.ipynb). Building one means assembling four layers, all of which live in the
**`dt-core`** repository (which is included here as a git submodule under `packages/dt-core`):

| Layer | Where it lives | What it does |
|-------|----------------|--------------|
| **Launcher** | `dt-core/launchers/<name>.sh` | The entrypoint `dts devel run -L` executes; calls `roslaunch`. |
| **Demo launch file** | `dt-core/.../duckietown_demos/launch/<name>.launch` | Turns on the subsystems this demo needs. |
| **`master.launch`** | `dt-core/.../duckietown_demos/launch/master.launch` | The one big launch file everything is built from. |
| **FSM config** | `dt-core/.../fsm/config/fsm_node/<name>.yaml` | The state machine for this demo (notebook 05). |

We will build the launcher and the demo launch file from scratch and wire them to `master.launch`, then
generate the files into the `dt-core` submodule so the new demo can actually be launched with `dts devel run`.

## Why everything is built off `master.launch`

Rather than have each demo wire up nodes from scratch, Duckietown puts **every** subsystem into a single
`master.launch` file, each wrapped in a switch:

```xml
<arg name="lane_following" default="false"/>
...
<group if="$(arg lane_following)">
    <!-- line detector, ground projection, lane filter, lane controller ... -->
</group>
```

Every subsystem defaults to **off**. A *demo* is then just a thin launch file that `include`s
`master.launch` and flips on the handful of switches it needs. This means:

* all the node wiring (topic remappings, parameter files) lives in **one** reviewed place, and
* a demo file stays short and readable, it is essentially a list of "which subsystems do I want?".

The required `demo_name` argument is the glue: it is passed into `master.launch` and, by default, also
selects the FSM configuration file (`fsm_file_name` defaults to `demo_name`).

## Locating the `dt-core` submodule

First, let's find the three directories we will be working with. The cell walks up from the notebook to
locate the `dt-core` submodule, so the rest of the notebook works regardless of where it is run from.

In [ ]:
import os

def find_dt_core(start="."):
    """Walk up from `start` looking for the dt-core submodule checkout."""
    here = os.path.abspath(start)
    candidates = [
        # running from notebooks/ (the usual case)
        "../packages/dt-core",
        # running from the repo root
        "packages/dt-core",
        "lx/lx-lane-following/packages/dt-core",
    ]
    for rel in candidates:
        p = os.path.abspath(os.path.join(here, rel))
        if os.path.isdir(os.path.join(p, "packages", "duckietown_demos")):
            return p
    raise FileNotFoundError("Could not locate the dt-core submodule from " + here)

DT_CORE = find_dt_core()
DEMOS_LAUNCH_DIR = os.path.join(DT_CORE, "packages", "duckietown_demos", "launch")
LAUNCHERS_DIR    = os.path.join(DT_CORE, "launchers")
FSM_CONFIG_DIR   = os.path.join(DT_CORE, "packages", "fsm", "config", "fsm_node")

print("dt-core:        ", DT_CORE)
print("demo launches:  ", os.path.relpath(DEMOS_LAUNCH_DIR, DT_CORE))
print("launchers:      ", os.path.relpath(LAUNCHERS_DIR, DT_CORE))
print("fsm configs:    ", os.path.relpath(FSM_CONFIG_DIR, DT_CORE))
assert os.path.exists(os.path.join(DEMOS_LAUNCH_DIR, "master.launch")), "master.launch not found!"
print("\nFound master.launch \u2713")

## Step 1 — see what subsystems are available

`master.launch` declares one `arg` per switch. Let's parse it and list the switches you can turn on. The
top-level switches (like `lane_following`, `fsm`, `LED`) enable a whole subsystem; the slash-prefixed ones
(like `/lane_following/lane_controller`) toggle individual nodes inside a subsystem.

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse(os.path.join(DEMOS_LAUNCH_DIR, "master.launch"))
root = tree.getroot()

# the switches are the top-level <arg> entries that default to true/false
switches = []
for arg in root.findall("arg"):
    default = (arg.get("default") or "").strip().lower()
    if default in ("true", "false"):
        switches.append(arg.get("name"))

subsystems = [s for s in switches if not s.startswith("/")]
sub_nodes  = [s for s in switches if s.startswith("/")]

print(f"{len(subsystems)} subsystem switches:")
for s in subsystems:
    print("  ", s)
print(f"\n{len(sub_nodes)} per-node switches (examples):")
for s in sub_nodes[:8]:
    print("  ", s)

## Step 2 — write the demo launch file

A demo launch file is short: it sets `demo_name`, includes `master.launch`, and flips on the switches it
needs. Here is the helper that builds one. Compare its output with the existing
[`lane_following.launch`](../packages/dt-core/packages/duckietown_demos/launch/lane_following.launch),
they have the same shape.

Note the optional `fsm_file_name`: by default `master.launch` looks for an FSM config named after the demo.
If your demo reuses an existing state machine you can point at it explicitly (we do that below so the example
demo works without you having to also build a new FSM config, see notebook 05 for that).

In [ ]:
def build_demo_launch(demo_name, switches, param_file_name="default",
                      visualization=True, fsm_file_name=None):
    lines = []
    lines.append('<?xml version="1.0" encoding="utf-8"?>')
    lines.append('<launch>')
    lines.append('    <arg name="veh" default="$(env VEHICLE_NAME)"/>')
    lines.append(f'    <arg name="demo_name" value="{demo_name}"/>')
    lines.append('')
    lines.append('    <include file="$(find duckietown_demos)/launch/master.launch">')
    lines.append('        <arg name="veh" value="$(arg veh)"/>')
    lines.append('        <arg name="demo_name" value="$(arg demo_name)"/>')
    lines.append(f'        <arg name="param_file_name" value="{param_file_name}" />')
    if fsm_file_name is not None:
        lines.append(f'        <arg name="fsm_file_name" value="{fsm_file_name}" />')
    lines.append(f'        <arg name="visualization" value="{str(visualization).lower()}" />')
    lines.append('')
    for sw in switches:
        # quote slash-prefixed names exactly as master.launch declares them
        lines.append(f'        <arg name="{sw}" value="true"/>')
    lines.append('    </include>')
    lines.append('</launch>')
    return "\n".join(lines) + "\n"

# Our example: a lane-following variant. Same subsystems as lane_following,
# reusing the existing 'lane_following' FSM config.
DEMO_NAME = "lane_following_slow"
DEMO_SWITCHES = [
    "fsm",
    "anti_instagram",
    "lane_following",
    "/lane_following/line_detection",
    "/lane_following/ground_projection",
    "/lane_following/lane_filter",
    "/lane_following/lane_controller",
    "LED",
    "/LED/emitter",
]

demo_launch_xml = build_demo_launch(DEMO_NAME, DEMO_SWITCHES, fsm_file_name="lane_following")
print(demo_launch_xml)

Before writing anything, let's make sure every switch we picked actually exists in `master.launch`, a typo
here would silently do nothing (the `group if` simply would not match).

In [ ]:
unknown = [s for s in DEMO_SWITCHES if s not in switches]
if unknown:
    print("These switches are NOT defined in master.launch:")
    for s in unknown:
        print("  ", s)
else:
    print("All", len(DEMO_SWITCHES), "switches are valid master.launch args \u2713")

## Step 3 — write the launcher

The launcher is the script `dts devel run` executes. It is a thin bash wrapper that `roslaunch`es the demo
launch file. The pattern (boilerplate + a single `dt-exec roslaunch` line) matches the existing
[`lane-following.sh`](../packages/dt-core/launchers/lane-following.sh).

**Naming convention:** launcher files use hyphens (`lane-following.sh`) while launch files use underscores
(`lane_following.launch`). The launcher's base name (without `.sh`) is what you pass to `dts devel run -L`.

In [ ]:
def build_launcher(launch_file_basename):
    return f'''#!/bin/bash

source /environment.sh

# initialize launch file
dt-launchfile-init

# YOUR CODE BELOW THIS LINE
# ----------------------------------------------------------------------------

# launching app
dt-exec roslaunch --wait duckietown_demos {launch_file_basename}.launch

# ----------------------------------------------------------------------------
# YOUR CODE ABOVE THIS LINE

# wait for app to end
dt-launchfile-join
'''

# launcher name uses hyphens; it roslaunches the underscore-named .launch file
LAUNCHER_NAME = DEMO_NAME.replace("_", "-")  # 'lane-following-slow'
launcher_sh = build_launcher(DEMO_NAME)
print(launcher_sh)

## Step 4 — generate the files into `dt-core`

Now we write the two files to their real locations in the submodule. The cell refuses to overwrite an
existing file (so you don't clobber a shipped demo), set `OVERWRITE = True` if you really mean to. The
launcher is made executable, just like the others.

In [ ]:
import stat

OVERWRITE = False

def write_file(path, contents, executable=False):
    if os.path.exists(path) and not OVERWRITE:
        print(f"SKIP (exists): {path}")
        return
    with open(path, "w") as f:
        f.write(contents)
    if executable:
        st = os.stat(path)
        os.chmod(path, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    print(f"WROTE: {path}")

launch_path   = os.path.join(DEMOS_LAUNCH_DIR, f"{DEMO_NAME}.launch")
launcher_path = os.path.join(LAUNCHERS_DIR, f"{LAUNCHER_NAME}.sh")

write_file(launch_path, demo_launch_xml)
write_file(launcher_path, launcher_sh, executable=True)

## Step 5 — validate the whole chain

Finally, let's confirm the pieces are consistent: the launcher must reference a launch file that exists, the
launch file must include `master.launch`, and we warn if the FSM config the demo points at is missing (the
node would fail to start on the robot otherwise).

In [ ]:
import re

def validate_demo(launcher_path, fsm_name=None):
    problems, notes = [], []
    if not os.path.exists(launcher_path):
        problems.append(f"launcher not found: {launcher_path}")
        return problems, notes
    launcher = open(launcher_path).read()
    m = re.search(r"roslaunch.*duckietown_demos\s+(\S+)\.launch", launcher)
    if not m:
        problems.append("launcher does not roslaunch a duckietown_demos launch file")
        return problems, notes
    launch_base = m.group(1)
    notes.append(f"launcher -> {launch_base}.launch")
    lp = os.path.join(DEMOS_LAUNCH_DIR, launch_base + ".launch")
    if not os.path.exists(lp):
        problems.append(f"launch file referenced by launcher is missing: {lp}")
        return problems, notes
    launch_xml = open(lp).read()
    if "master.launch" not in launch_xml:
        problems.append(f"{launch_base}.launch does not include master.launch")
    # which FSM config will this demo use?
    fsm = fsm_name
    if fsm is None:
        fm = re.search(r'fsm_file_name"\s+value="([^"]+)"', launch_xml)
        fsm = fm.group(1) if fm else launch_base   # defaults to demo_name
    fsm_path = os.path.join(FSM_CONFIG_DIR, fsm + ".yaml")
    if os.path.exists(fsm_path):
        notes.append(f"FSM config -> {fsm}.yaml \u2713")
    else:
        notes.append(f"FSM config -> {fsm}.yaml  (MISSING - build one per notebook 05)")
    return problems, notes

problems, notes = validate_demo(launcher_path, fsm_name="lane_following")
for n in notes:
    print("  ", n)
print()
if problems:
    print("Problems found:")
    for p in problems:
        print("  -", p)
else:
    print("Demo chain looks consistent \u2713")

## Step 6 — build and run your demo

Creating the files is only half of it, now you build and run the demo using the same `dts devel` workflow
introduced in [notebook 00](00_DTS_devel_API.ipynb). All of these commands run from the `dt-core` project
directory:

```bash
cd packages/dt-core
```

**1. Build** so your new launcher and launch file are baked into the image (`ROBOT_NAME` is your real or
virtual robot):

```bash
dts devel build -H ROBOT_NAME
```

**2. Run** by passing the launcher's base name (no `.sh`) to `-L`:

```bash
dts devel run -H ROBOT_NAME -L lane-following-slow
```

That runs `launchers/lane-following-slow.sh`, which roslaunches `lane_following_slow.launch`, which includes
`master.launch` with your switches turned on, bringing up the FSM (using the `lane_following` config) and the
lane following stack. As in notebook 00, your robot's LEDs turn green when it is ready to drive.

> **Handy flags (see notebook 00 for the full list):**
> * If you only changed parameters or Python code (no recompile needed), skip the rebuild and sync your code
>   in instead: `dts devel run -H ROBOT_NAME -L lane-following-slow -s -M`.
> * Build locally instead of on the robot by omitting `-H`, then connect with `dts devel run -R ROBOT_NAME`.
> * Add `-f` to force a build even with uncommitted local changes.

Because the example reuses the lane following stack and FSM config, `lane_following_slow` behaves just like
lane following, the point is the *scaffolding*: change the switches, the param files, or the FSM config and
you have a brand new demo.

## Recap & exercise

Building a demo is assembling four layers, all in the `dt-core` submodule:

1. **`master.launch`** already contains every subsystem behind a switch, you rarely touch it.
2. A **demo launch file** includes `master.launch` and flips on the switches you need.
3. A **launcher** (`.sh`) roslaunches that demo launch file; its base name is what you pass to `dts devel run -L`.
4. An **FSM config** (notebook 05) defines the states; by default it shares the demo's name.

### Exercise

Build your own demo end-to-end:

1. Pick a `DEMO_NAME` and a set of switches (try adding `obstacle_detection` and
   `/obstacle_detection/tof` on top of the lane following stack).
2. Create a matching FSM config `config/fsm_node/<DEMO_NAME>.yaml` using what you learned in notebook 05
   (give it a `LANE_FOLLOWING` and a `STOP` state driven by the obstacle events), and **remove** the
   `fsm_file_name` override so the demo uses *your* config.
3. Re-run Steps 2-5. The validator should report your launcher, launch file, and FSM config all line up.

When `validate_demo` prints no problems and resolves to *your* FSM config, you've built a demo from scratch.